# Escada de ablação — paridade de motor (issue #93)

Um degrau, um componente. O **primeiro vermelho** aponta o culpado — nada depois dele é confiável (a divergência cascateia). Este notebook só lê os `results/L*_*.json`; toda métrica saiu de `policy.simulate().data`.

**Base única** (SHA-256 no rodapé), `calibration_bins=10` fixo, `current_approval_col` ligado desde o L0 (swap-in existe em todo degrau).

In [ ]:
import os
import pandas as pd
import ladder_common as lc

TAG = os.environ.get('LADDER_TAG', '1p5m')  # final report uses 1.5M
recs = lc.load_results(TAG)  # reads results/*.jsonl (gitignored, regenerable)
shas = {r['base_sha256'] for r in recs.values()}
print('base SHA-256 (deve ser UM):', shas)
print('n:', {r['n'] for r in recs.values()}, ' TAG:', TAG)

## Matriz degrau × invariante — verde/vermelho
Tolerância declarada por degrau **antes** de rodar (`ladder_common.TOL`).

In [ ]:
def paint(v): return '🟩' if v == 'green' else '🟥'
rows = []
first_red = None
for step in lc.STEPS:
    m = recs[(step, 'main')]['measured']
    v = recs[(step, 'v05')]['measured']
    for r in lc.compare(m, v, step):
        if r['verdict'] == 'red' and first_red is None:
            first_red = (step, r['invariant'])
        rows.append({'degrau': step, 'adiciona': lc.STEP_ADDS[step],
                     'invariante': r['invariant'], 'tol': r['tolerance'],
                     'delta': r['delta'], 'veredito': paint(r['verdict'])})
matrix = pd.DataFrame(rows)
print('PRIMEIRO DEGRAU VERMELHO:', first_red)
matrix

## Placar por degrau (um olhar)

In [ ]:
board = []
for step in lc.STEPS:
    m = recs[(step, 'main')]['measured']
    v = recs[(step, 'v05')]['measured']
    cmp = lc.compare(m, v, step)
    reds = [r['invariant'] for r in cmp if r['verdict'] == 'red']
    board.append({'degrau': step, 'adiciona': lc.STEP_ADDS[step],
                  'veredito': '🟩 verde' if not reds else '🟥 ' + ', '.join(reds)})
pd.DataFrame(board)

## Tabela-resumo por decil (v0.5, degrau selecionado)
Decis da calibração nas linhas; keep-in vs swap-in em volume e inad, taxa de conversão efetiva, e a razão `swap_in_inad / keep_bin_pd`.

**Sanity #1** — carteira = SOMARPRODUTO das células ≡ `blended_default`.
**Sanity #2** — sob stress, a razão swap/keep = fator, decil a decil.

In [ ]:
STEP = os.environ.get('LADDER_STEP', 'L4')  # L4 = primeiro degrau com stress
m = recs[(STEP, 'v05')]['measured']
tbl = pd.DataFrame(m['decile_table'])
print(f'degrau {STEP} — {lc.STEP_ADDS[STEP]}')
s = m['sanity']
print('sumproduct == blended :', s['sumproduct_matches_blended'],
      f"({s['sumproduct_default']:.6f} vs {m['blended_default']:.6f})")
print('razão swap/keep == fator', s['expected_ratio'], ':', s['stress_ratio_ok'])
tbl.round(4)

## Diff mínimo do primeiro degrau vermelho
Config + contadores divergentes + delta — suficiente pra abrir bug sem re-rodar.

In [ ]:
if first_red:
    step = first_red[0]
    m = recs[(step, 'main')]['measured']; v = recs[(step, 'v05')]['measured']
    print('degrau', step, '—', lc.STEP_ADDS[step])
    print('approval (pre-rate)  main', m['approval'], 'v05', v['approval'], '(deve bater)')
    print('quadrantes iguais    :', m['quadrant_counts'] == v['quadrant_counts'])
    print('contracted           main', round(m['contracted']), 'v05', round(v['contracted']),
          'delta', round(m['contracted'] - v['contracted']))
    print('blended_default      main', round(m['blended_default'], 5), 'v05', round(v['blended_default'], 5))
    mk = sum(r['keep_in_vol_decision'] for r in m['decile_table']); vk = sum(r['keep_in_vol_decision'] for r in v['decile_table'])
    ms = sum(r['swap_in_vol'] for r in m['decile_table']); vs = sum(r['swap_in_vol'] for r in v['decile_table'])
    print('keep_in_vol (decisão) main', round(mk), 'v05', round(vk), '<-- fonte da divergência' if abs(mk-vk)>1 else '')
    print('swap_in_vol          main', round(ms), 'v05', round(vs))
else:
    print('paridade total — nenhum degrau vermelho.')

## L8 — idiomas exclusivos de cada engine (documentação, NÃO degraus)

In [ ]:
for eng in ('main', 'v05'):
    notes = recs[('L5', eng)].get('l8_engine_exclusive_notes', {})
    print('===', eng, '===')
    for k, val in notes.items():
        print(f'  {k}: {val}')

print('\n--- warnings capturados (nunca suprimidos) ---')
for step in lc.STEPS:
    for eng in ('main', 'v05'):
        w = recs[(step, eng)]['warnings']
        if w:
            print(step, eng, w)